# Surface-point extraction

This notebook compares two ways to classify query points relative to an oriented surface and demonstrates boundary extraction from a volumetric point cloud.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import seaborn as sns
from matplotlib import patches
from scipy import interpolate, spatial
from tqdm.auto import tqdm

In [ ]:
%config InlineBackend.figure_format = 'retina'

In [ ]:
RNG = np.random.default_rng(42)

## Visualization helpers


In [ ]:
def configure_plot_style(*, reset: bool = False) -> None:
    """Configure a consistent plotting style."""
    if reset:
        sns.reset_defaults()
        return
    sns.set_theme(
        context="notebook",
        style="white",
        palette="colorblind",
        rc={
            "xtick.bottom": True,
            "xtick.color": "black",
            "xtick.direction": "in",
            "ytick.direction": "in",
        },
    )

In [ ]:
def set_3d_params(
    ax: plt.Axes,
    aspect: tuple[float, float, float] = (1, 1, 1),
) -> plt.Axes:
    """Configure labels, ticks, panes, and aspect ratio for 3D axes."""
    ax.set(xlabel="x", ylabel="y", zlabel="z")
    ax.xaxis.set_major_locator(plt.MaxNLocator(3))
    ax.yaxis.set_major_locator(plt.MaxNLocator(3))
    ax.zaxis.set_major_locator(plt.MaxNLocator(3))
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.set_box_aspect(aspect)
    return ax

## Input data


In [ ]:
def generate_2d_gaussian(
    x: float | np.ndarray,
    y: float | np.ndarray,
    *,
    amplitude: float = 1,
    x_center: float = 0,
    y_center: float = 0,
    x_scale: float = 1,
    y_scale: float = 1,
) -> float | np.ndarray:
    """Evaluate a two-dimensional Gaussian function."""
    exponent = -(
        (x - x_center) ** 2 / (2 * x_scale**2) + (y - y_center) ** 2 / (2 * y_scale**2)
    )
    return amplitude * np.exp(exponent)

In [ ]:
# Generate the 2D Gaussian
x = np.linspace(-1, 1, 51)
y = np.linspace(-1, 1, 51)
X, Y = np.meshgrid(x, y)
Z = generate_2d_gaussian(X, Y, amplitude=2, x_scale=0.3, y_scale=0.3)

In [ ]:
# Plot the surface
configure_plot_style()
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax = set_3d_params(ax)
surf = ax.plot_surface(X, Y, Z, lw=0, cstride=1, rstride=1, antialiased=False)

In [ ]:
def estimate_normals(points: np.ndarray, k: int) -> np.ndarray:
    """Estimate point normals with PCA on local neighborhoods."""
    tree = spatial.KDTree(points)
    normals = np.empty_like(points)
    for index, point in enumerate(points):
        _, neighbor_indices = tree.query(point, k=k, eps=0.1, workers=-1)
        neighborhood = points[np.atleast_1d(neighbor_indices)]
        centered = neighborhood - np.mean(neighborhood, axis=0)
        covariance = centered.T @ centered
        _, eigenvectors = np.linalg.eigh(covariance)
        normals[index] = eigenvectors[:, 0]
    return normals

In [ ]:
def orient_normals(
    points: np.ndarray,
    normals: np.ndarray,
    k: int = 20,
    *,
    convex: bool = False,
) -> np.ndarray:
    """Orient point normals consistently or away from the centroid."""
    if convex:
        radial_vectors = points - np.mean(points, axis=0)
        inward = np.einsum("ij,ij->i", normals, radial_vectors) < 0
        oriented = normals.copy()
        oriented[inward] *= -1
        return oriented

    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(points)
    point_cloud.normals = o3d.utility.Vector3dVector(normals)
    point_cloud.orient_normals_consistent_tangent_plane(k)
    return np.asarray(point_cloud.normals)

In [ ]:
# Create the point cloud and generate a unit normal at each point
points = np.c_[X.ravel(), Y.ravel(), Z.ravel()]
normals = estimate_normals(points, k=20)
normals = orient_normals(points, normals, k=20)

In [ ]:
# Plot the surface with normals
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax = set_3d_params(ax)
surf = ax.plot_surface(X, Y, Z, lw=0, cstride=1, rstride=1, antialiased=False)
q = ax.quiver(
    *points.T,
    *normals.T,
    color="k",
    lw=0.5,
    length=0.25,
    arrow_length_ratio=0.15,
)

## Assessment of the location of a query point relative to the point cloud


### RBF interpolation


The first method approximates a signed-distance function (SDF) from oriented surface points and interpolates it with a radial basis function. It follows four steps:

1. Define the query point, $p$.
2. Sample points on both sides of the surface along its normals.
3. Interpolate the sampled SDF values with a radial basis function.
4. Evaluate the interpolant at $p$; positive values indicate points outside the surface.

The optional [`polatory`](https://github.com/polatory/polatory) implementation from the original experiment is retained below. If it is unavailable, the following section provides a SciPy-only implementation.


In [ ]:
try:
    import polatory
except ModuleNotFoundError:
    print("Optional dependency 'polatory' is not installed; using SciPy below.")
else:
    point_out = np.array([-1, -1, 1])
    pairwise_distances = spatial.distance.pdist(points)
    sdf = polatory.SdfDataGenerator(
        points,
        normals,
        np.min(pairwise_distances),
        np.max(pairwise_distances),
    )
    keep = polatory.DistanceFilter(sdf.sdf_points, 1e-4).filtered_indices
    sdf_points = sdf.sdf_points[keep]
    sdf_values = sdf.sdf_values[keep]

    model = polatory.Model(
        polatory.Biharmonic3D([1.0]),
        poly_dimension=2,
        poly_degree=1,
    )
    interpolator = polatory.Interpolant(model)
    interpolator.fit(sdf_points, sdf_values, absolute_tolerance=1e-4)
    signed_distance = interpolator.evaluate(point_out)
    location = "OUTSIDE" if signed_distance > 0 else "INSIDE"
    print(f"The point is {location} the Gaussian surface.")

The implementation below handles multiple query points at once.


In [ ]:
MIN_SIZE = 10
PROBABILITY_THRESHOLD = 0.5

In [ ]:
def assess_position(
    query_points: np.ndarray,
    evaluation_points: np.ndarray,
    normals: np.ndarray | None = None,
    k: int | None = None,
) -> np.ndarray:
    """Approximate signed distances for query points with RBF interpolation."""
    size = evaluation_points.shape[0]
    if size < MIN_SIZE:
        msg = f"Number of points must be at least {MIN_SIZE}"
        raise ValueError(msg)

    if normals is None:
        k = k or min(max(int(2 * np.log(size)), 5), 30)
        normals = estimate_normals(evaluation_points, k)
        normals = orient_normals(evaluation_points, normals, k)
    unit_normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)

    tree = spatial.KDTree(evaluation_points)
    nearest_distances, _ = tree.query(evaluation_points, k=2)
    offset = float(np.median(nearest_distances[:, 1]))
    sdf_points = np.vstack(
        [
            evaluation_points,
            evaluation_points + offset * unit_normals,
            evaluation_points - offset * unit_normals,
        ],
    )
    sdf_values = np.concatenate(
        [
            np.zeros(size),
            np.full(size, offset),
            np.full(size, -offset),
        ],
    )

    interpolator = interpolate.RBFInterpolator(
        sdf_points,
        sdf_values,
        kernel="linear",
        degree=1,
        neighbors=min(100, sdf_points.shape[0]),
    )
    return interpolator(np.atleast_2d(query_points))

In [ ]:
# Define query points to test an assessment function
query_points = np.c_[X.ravel(), Y.ravel(), np.ones_like(X).ravel()]

In [ ]:
# Plot query points in 3D
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.contourf(X, Y, Z, zdir="y", offset=1, levels=1, colors="b")
ax.contourf(X, Y, Z, zdir="x", offset=-1, levels=1, colors="b")
ax.scatter(*query_points.T, fc="orange", ec="k", s=5, lw=0.5)
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Assess the query points and show those outside the surface.
signed_distances = assess_position(query_points, points, normals=normals)
outside = signed_distances > 0

fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.contourf(X, Y, Z, zdir="y", offset=1, levels=1, colors="b")
ax.contourf(X, Y, Z, zdir="x", offset=-1, levels=1, colors="b")
ax.scatter(*query_points[outside].T, fc="w", ec="k", s=15, lw=0.5)
ax = set_3d_params(ax)
ax.view_init(25, -70)

# Compare the xy projection with the analytic f(x, y) = 1 contour.
indices = np.where(np.isclose(Z, 1, rtol=1e-2, atol=1e-2))
radius = np.mean(np.sqrt(X[indices] ** 2 + Y[indices] ** 2))
fig, ax = plt.subplots(figsize=(4, 4))
circle = patches.Circle((0, 0), radius, fc="none", ec="k")
ax.add_patch(circle)
ax.scatter(*query_points[outside, :2].T, fc="w", ec="k", s=7, lw=0.5)
ax.set(xlabel="x", ylabel="y")

### Normal direction


The second method uses the direction of the oriented normals:

1. Define the query point, $p$.
2. Find the $k$ surface points nearest to $p$.
3. Compute the dot product between each relative position vector and its corresponding unit normal, $\hat{\mathbf{n}}_i$:

$$
(\mathbf{p} - \mathbf{x}_i) \cdot \hat{\mathbf{n}}_i
$$

4. Classify the point as outside when at least half of the dot products are positive.


In [ ]:
# Step 1
point_out = np.array([1, -1, 2])  # out of the point cloud

In [ ]:
# Show the point in 3D
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
surf = ax.plot_surface(X, Y, Z, lw=0, cstride=1, rstride=1, antialiased=False)
ax.scatter(*points.T, fc="w", ec="k", s=5, lw=0.5)
ax.scatter(*point_out, fc="orange", ec="k", s=15, lw=0.5)
ax.text(*point_out + np.array([0, 0, 0.2]), f"{point_out}")
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Step 2
tree = spatial.KDTree(points)
dist, idx = tree.query(point_out, k=20)

In [ ]:
# Show the point in 3D with neighbors
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.scatter(*np.delete(points, idx, axis=0).T, fc="w", ec="k", s=5, lw=0.5)
ax.scatter(*points[idx, ...].T, fc="green", ec="k", s=15, lw=0.5)
ax.scatter(*point_out, fc="orange", ec="k", s=15, lw=0.5)
ax.text(*point_out + np.array([0, 0, 0.2]), f"{point_out}")
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Show the point in 3D with neighbors and normals
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.scatter(*np.delete(points, idx, axis=0).T, fc="w", ec="k", s=5, lw=0.5)
ax.scatter(*points[idx, ...].T, fc="green", ec="k", s=15, lw=0.5)
ax.quiver(
    *points[idx, ...].T,
    *normals[idx, ...].T,
    color="k",
    lw=0.5,
    length=0.5,
    arrow_length_ratio=0.15,
)
ax.scatter(*point_out, fc="orange", ec="k", s=15, lw=0.5)
ax.scatter(0, 0, 0, fc="k", ec="k", s=15, lw=0.5)
ax.quiver(0, 0, 0, *point_out, color="k", lw=1, arrow_length_ratio=0.1)
ax.text(*point_out + np.asarray([0, 0, 0.2]), f"{point_out}")
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Step 3
prod = np.sum((point_out - points[idx]) * normals[idx], axis=1)

In [ ]:
# Step 4
prob_threshold = 0.5
prob = np.sum(prod > 0) / prod.size
if prob > prob_threshold:
    print(f"The point is OUT of the point cloud ({prob:.2f})")
else:
    print(f"The point is WITHIN the point cloud ({1 - prob:.2f})")

This method is faster and simpler than SDF interpolation, but it is sensitive to the number of neighboring surface points and may be unreliable for complex boundaries.


In [ ]:
def assess_position_by_normals(
    query_points: np.ndarray,
    evaluation_points: np.ndarray,
    sample_count: int = 5,
    normals: np.ndarray | None = None,
    k: int | None = None,
) -> np.ndarray:
    """Classify query points using nearby oriented surface normals."""
    size = evaluation_points.shape[0]
    if size < MIN_SIZE:
        msg = f"Number of points must be at least {MIN_SIZE}"
        raise ValueError(msg)

    if normals is None:
        k = k or min(max(int(2 * np.log(size)), 5), 30)
        normals = estimate_normals(evaluation_points, k)
        normals = orient_normals(evaluation_points, normals, k)
    unit_normals = normals / np.linalg.norm(normals, axis=1, keepdims=True)

    tree = spatial.KDTree(evaluation_points)
    _, indices = tree.query(
        np.atleast_2d(query_points),
        k=sample_count,
        workers=-1,
    )
    closest_points = evaluation_points[indices]
    relative_positions = np.atleast_2d(query_points)[:, np.newaxis, :] - closest_points
    points_outside = (
        np.einsum("ijk,ijk->ij", relative_positions, unit_normals[indices]) > 0
    )
    return np.mean(points_outside, axis=1) >= PROBABILITY_THRESHOLD

In [ ]:
# Define query points to test an assessment function
query_points = np.c_[X.ravel(), Y.ravel(), np.ones_like(X).ravel()]

In [ ]:
# Plot query points in 3D
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.contourf(X, Y, Z, zdir="y", offset=1, levels=1, colors="b")
ax.contourf(X, Y, Z, zdir="x", offset=-1, levels=1, colors="b")
ax.scatter(*query_points.T, fc="orange", ec="k", s=5, lw=0.5)
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Check if the orange dots are inside or out of the point cloud
out = assess_position_by_normals(query_points, points, sample_count=5, normals=normals)

In [ ]:
# Plot query points in 3D with assessment
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.contourf(X, Y, Z, zdir="y", offset=1, levels=1, colors="b")
ax.contourf(X, Y, Z, zdir="x", offset=-1, levels=1, colors="b")
ax.scatter(*query_points[out, ...].T, fc="w", ec="k", s=15, lw=0.5)
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Find out the approximate radius where f(x, y) is ~1
idx = np.where(np.isclose(Z, 1, rtol=1e-2, atol=1e-2))
r = np.mean(np.sqrt(X[idx] ** 2 + Y[idx] ** 2))

In [ ]:
# Plot only the xy-projection of the outside points
fig = plt.figure(figsize=(4, 4))
ax = plt.axes()
circle = patches.Circle((0, 0), r, fc="none", ec="k")
ax.add_patch(circle)
ax.scatter(*query_points[out, :2].T, fc="w", ec="k", s=7, lw=0.5)
ax.set(xlabel="x", ylabel="y");

## Extraction of points on the boundary of the point cloud


Assume a set of points, $\mathbb{X} = \{\mathbf{x}_1, \mathbf{x}_2, \dots, \mathbf{x}_n\}$, samples a compact region $\Omega \subset \mathbb{R}^3$. The goal is to identify the subset on the boundary $S = \partial \Omega$, called *surface points*.


<div style="text-align:center">
    <img style="margin:20px; width:450px;" src="../../media/pc-surf.svg">
</div>


The following steps should be applied to each point, $\mathbf{x}_i$, in $\mathbb{X}$.


<div style="text-align:center">
    <img style="margin:20px; width:750px;" src="../../media/pc-surf-extract.svg">
</div>


The implementation below uses `scipy.spatial`.


In [ ]:
def extract_surface_points(points: np.ndarray, radius: float) -> np.ndarray:
    """Extract points near the boundary of a volumetric point cloud."""
    surface_indices: list[int] = []
    tree = spatial.KDTree(points)
    for index, point in enumerate(tqdm(points)):
        # Step 1: extract a local neighborhood around the query point.
        neighbor_indices = tree.query_ball_point(point, r=radius)
        neighborhood = points[neighbor_indices]

        # Step 2: estimate the local normal direction.
        centered = neighborhood - np.mean(neighborhood, axis=0)
        covariance = centered.T @ centered
        _, eigenvectors = np.linalg.eigh(covariance)
        normal = eigenvectors[:, 0]

        # Step 3: check for an empty ball on either side of the tangent plane.
        centers = [point + normal * radius / 2, point - normal * radius / 2]
        if any(
            len(tree.query_ball_point(center, r=radius / 2)) <= 1 for center in centers
        ):
            surface_indices.append(index)

    return points[surface_indices]

This toy example extracts surface points from samples distributed uniformly inside the unit ball.


In [ ]:
def sample_ball(num_points: int) -> np.ndarray:
    """Sample points uniformly from the unit ball."""
    directions = RNG.normal(size=(num_points, 3))
    directions /= np.linalg.norm(directions, axis=1, keepdims=True)
    radii = RNG.random(num_points) ** (1 / 3)
    return directions * radii[:, np.newaxis]

In [ ]:
# Generate points within a ball of radius 1
points = sample_ball(9_999)

In [ ]:
# Show the original point cloud
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.scatter(*points.T, s=1, alpha=0.25)
ax = set_3d_params(ax)
ax.view_init(25, -70);

In [ ]:
# Extract the surface points
surface_points = extract_surface_points(points, radius=0.5)

In [ ]:
# Show the extracted surface points
fig = plt.figure(figsize=(5, 5))
ax = plt.axes(projection="3d")
ax.scatter(*surface_points.T, s=1)
ax = set_3d_params(ax)
ax.view_init(25, -70);